In [ ]:
from phytclust.main import PhytClust
from phytclust import plot_tree
from phytclust import pairwise_distances
import io
import os
from Bio import Phylo
from ete3 import Tree
import random
from ete3 import Tree, TreeStyle
import string

ts = TreeStyle()
ts.show_leaf_name = True
ts.scale = 20

Import and plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters for the Gaussian distribution
mean = 0
std_dev = 0.4

# Generate data points from the Gaussian distribution
data = np.random.normal(mean, std_dev, 1000)

# Plot the histogram
plt.hist(data, bins=30, density=True, alpha=0.6, color="g")

# Plot the Gaussian distribution curve
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = np.exp(-((x - mean) ** 2) / (2 * std_dev**2)) / (std_dev * np.sqrt(2 * np.pi))
plt.plot(x, p, "k", linewidth=2)

title = "Gaussian Distribution\nMean = 0, Standard Deviation = 0.1"
plt.title(title)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import os
import json
import pandas as pd
import plotly.express as px

# Define the base directory containing tree folders
base_dir = (
    "/home/ganesank/project/phytclust/simulations/phytclust_results_all_k_old_score"
)

# Initialize a list to hold all parsed data
all_parsed_data = []

# Iterate through each tree folder
for tree_folder in os.listdir(base_dir):
    tree_folder_path = os.path.join(base_dir, tree_folder)
    if os.path.isdir(tree_folder_path):
        comparison_file_path = os.path.join(tree_folder_path, "comparison_results.json")
        if os.path.exists(comparison_file_path):
            # Load the JSON data
            with open(comparison_file_path, "r") as file:
                data = json.load(file)

            # Filter out entries with both 'ARI' and 'Tree'
            filtered_data = [
                entry for entry in data if "ARI" in entry and "Tree" in entry
            ]

            # Parse the 'Tree' string to extract outliers, sampling, and noise values
            for entry in filtered_data:
                tree_info = entry["Tree"]
                parts = tree_info.split("_")
                if len(parts) >= 6:  # Adjusted to handle longer parts
                    outliers = int(parts[1])
                    sampling = int(parts[3])
                    noise = int(parts[5])
                    all_parsed_data.append(
                        {
                            "Tree": tree_info,
                            "ARI": entry["ARI"],
                            "percentage_outliers": outliers,
                            "sampling_fraction": sampling,
                            "noise_level": noise,
                        }
                    )

# Convert all parsed data to DataFrame
df = pd.DataFrame(all_parsed_data)

# Ensure that the data types are correct
df["percentage_outliers"] = df["percentage_outliers"].astype(int)
df["sampling_fraction"] = df["sampling_fraction"].astype(int)
df["noise_level"] = df["noise_level"].astype(int)
df["ARI"] = df["ARI"].astype(float)

# Group by percentage_outliers, sampling_fraction, and noise_level, and calculate the mean ARI
df_mean = (
    df.groupby(["percentage_outliers", "sampling_fraction", "noise_level"])
    .mean()
    .reset_index()
)

# Sort the DataFrame by noise_level
df_mean = df_mean.sort_values(by="noise_level")

# Create a heatmap with Plotly
fig_heatmap = px.density_heatmap(
    df_mean,
    x="percentage_outliers",
    y="sampling_fraction",
    z="ARI",
    facet_col="noise_level",
    color_continuous_scale="RdYlBu",  # Pre-built color scale: red to blue
    title="Mean ARI with Varying Outliers, Sampling, and Noise",
    labels={
        "percentage_outliers": "% Outliers",
        "sampling_fraction": "Sampling Fraction",
        "ARI": "Mean ARI",
        "noise_level": "Noise",
    },
    range_color=[0, 1],  # Set the color scale range for ARI
)

# Update the layout to change the legend title and adjust the width
fig_heatmap.update_layout(
    coloraxis_colorbar=dict(title="Mean ARI"), width=1200  # Adjust the width as needed
)

# Create a 3D scatter plot with Plotly
fig_3d = px.scatter_3d(
    df_mean,
    x="percentage_outliers",
    y="sampling_fraction",
    z="noise_level",
    color="ARI",
    color_continuous_scale="RdYlBu",  # Pre-built color scale: red to blue
    title="3D Scatter Plot of Mean ARI",
    labels={
        "percentage_outliers": "% Outliers",
        "sampling_fraction": "Sampling Fraction",
        "ARI": "Mean ARI",
        "noise_level": "Noise",
    },
    range_color=[0, 1],  # Set the color scale range for ARI
)

# Update the layout to change the legend title
fig_3d.update_layout(coloraxis_colorbar=dict(title="Mean ARI"))

# Show the plots
fig_heatmap.show()
fig_3d.show()

Runtime

In [ ]:
import os
import json
import pandas as pd
import plotly.express as px

# Define the base directory containing tree folders
base_dir = (
    "/home/ganesank/project/phytclust/simulations/PhyloPart_v2.1/phylopart_output"
)

# Initialize a list to hold all parsed data
all_parsed_data = []

# Iterate through each tree folder
for tree_folder in os.listdir(base_dir):
    tree_folder_path = os.path.join(base_dir, tree_folder)
    if os.path.isdir(tree_folder_path):
        comparison_file_path = os.path.join(tree_folder_path, "ari_results.json")
        if os.path.exists(comparison_file_path):
            # Load the JSON data
            with open(comparison_file_path, "r") as file:
                data = json.load(file)

            # Filter out entries with both 'ARI' and 'Tree'
            filtered_data = [
                entry for entry in data if "ARI" in entry and "Tree" in entry
            ]

            # Parse the 'Tree' string to extract outliers, sampling, and noise values
            for entry in filtered_data:
                tree_info = entry["Tree"]
                parts = tree_info.split("_")
                if len(parts) >= 6:  # Adjusted to handle longer parts
                    outliers = int(parts[1])
                    sampling = int(parts[3])
                    noise = int(parts[5])
                    all_parsed_data.append(
                        {
                            "Tree": tree_info,
                            "ARI": entry["ARI"],
                            "percentage_outliers": outliers,
                            "sampling_fraction": sampling,
                            "noise_level": noise,
                        }
                    )

# Convert all parsed data to DataFrame
df = pd.DataFrame(all_parsed_data)

# Ensure that the data types are correct
df["percentage_outliers"] = df["percentage_outliers"].astype(int)
df["sampling_fraction"] = df["sampling_fraction"].astype(int)
df["noise_level"] = df["noise_level"].astype(int)
df["ARI"] = df["ARI"].astype(float)

# Print the DataFrame to check the parsed data
# print("Parsed DataFrame:")
# print(df.head())

# Group by percentage_outliers, sampling_fraction, and noise_level, and calculate the mean ARI
df_mean = (
    df.groupby(["percentage_outliers", "sampling_fraction", "noise_level"])
    .mean()
    .reset_index()
)

# Print the grouped DataFrame to check the mean calculations
# print("Grouped DataFrame with Mean ARI:")
# print(df_mean.head())

# Sort the DataFrame by noise_level
df_mean = df_mean.sort_values(by="noise_level")

# Create a heatmap with Plotly
fig_heatmap = px.density_heatmap(
    df_mean,
    x="percentage_outliers",
    y="sampling_fraction",
    z="ARI",
    facet_col="noise_level",
    color_continuous_scale="RdBu",  # Pre-built color scale: red to blue
    title="Mean ARI with Varying Outliers, Sampling, and Noise",
    labels={
        "percentage_outliers": "% Outliers",
        "sampling_fraction": "Sampling Fraction",
        "ARI": "Mean ARI",
        "noise_level": "Noise",
    },
    range_color=[0, 1],  # Set the color scale range for ARI
)

# Update the layout to change the legend title and adjust the width
fig_heatmap.update_layout(
    coloraxis_colorbar=dict(title="Mean ARI"), width=1200  # Adjust the width as needed
)

# Create a 3D scatter plot with Plotly
fig_3d = px.scatter_3d(
    df_mean,
    x="percentage_outliers",
    y="sampling_fraction",
    z="noise_level",
    color="ARI",
    color_continuous_scale="RdYlBu",  # Pre-built color scale: red to blue
    title="3D Scatter Plot of Mean ARI",
    labels={
        "percentage_outliers": "% Outliers",
        "sampling_fraction": "Sampling Fraction",
        "ARI": "Mean ARI",
        "noise_level": "Noise",
    },
    range_color=[0, 1],  # Set the color scale range for ARI
)

# Update the layout to change the legend title
fig_3d.update_layout(coloraxis_colorbar=dict(title="Mean ARI"))

# Show the plots
fig_heatmap.show()
fig_3d.show()

In [ ]:
import os
import json
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio

# Define the base directory containing tree folders
base_dir = (
    "/home/ganesank/project/phytclust/simulations/phytclust_results_all_k_old_score"
)

# Initialize a list to hold all parsed data
all_parsed_data = []

# Iterate through each tree folder
for tree_folder in os.listdir(base_dir):
    tree_folder_path = os.path.join(base_dir, tree_folder)
    if os.path.isdir(tree_folder_path):
        comparison_file_path = os.path.join(tree_folder_path, "comparison_results.json")
        if os.path.exists(comparison_file_path):
            # Load the JSON data
            with open(comparison_file_path, "r") as file:
                data = json.load(file)

            # Filter out entries with both 'ARI' and 'Tree'
            filtered_data = [
                entry for entry in data if "ARI" in entry and "Tree" in entry
            ]

            # Parse the 'Tree' string to extract outliers, sampling, and noise values
            for entry in filtered_data:
                tree_info = entry["Tree"]
                parts = tree_info.split("_")
                if len(parts) >= 6:  # Adjusted to handle longer parts
                    outliers = int(parts[1])
                    sampling = int(parts[3])
                    noise = int(parts[5])
                    all_parsed_data.append(
                        {
                            "Tree": tree_info,
                            "ARI": entry["ARI"],
                            "percentage_outliers": outliers,
                            "sampling_fraction": sampling,
                            "noise_level": noise,
                        }
                    )

# Convert all parsed data to DataFrame
df = pd.DataFrame(all_parsed_data)

# Ensure that the data types are correct
df["percentage_outliers"] = df["percentage_outliers"].astype(int)
df["sampling_fraction"] = df["sampling_fraction"].astype(int)
df["noise_level"] = df["noise_level"].astype(int)
df["ARI"] = df["ARI"].astype(float)

# Group by percentage_outliers, sampling_fraction, and noise_level, and calculate the mean ARI
df_mean = (
    df.groupby(["percentage_outliers", "sampling_fraction", "noise_level"])
    .mean()
    .reset_index()
)

# Sort the DataFrame by noise_level
df_mean = df_mean.sort_values(by="noise_level")

# Create a grid layout for the heatmaps
num_heatmaps = len(df_mean["noise_level"].unique())
cols = 4  # Number of columns in the grid
rows = (num_heatmaps // cols) + (
    num_heatmaps % cols > 0
)  # Calculate the number of rows needed

fig = make_subplots(
    rows=rows,
    cols=cols,
    subplot_titles=[f"Noise Level: {nl}" for nl in df_mean["noise_level"].unique()],
    x_title="% Outliers",
    y_title="Sampling Fraction",
)

# Add each heatmap to the grid
for i, noise_level in enumerate(df_mean["noise_level"].unique()):
    df_noise = df_mean[df_mean["noise_level"] == noise_level]
    pivot_table = df_noise.pivot_table(
        values="ARI",
        index="sampling_fraction",
        columns="percentage_outliers",
        aggfunc="mean",
    )

    heatmap = go.Heatmap(
        z=pivot_table.values,
        x=pivot_table.columns,
        y=pivot_table.index,
        colorscale="RdYlBu",
        zmin=0,
        zmax=1,
        colorbar=dict(title="Mean ARI"),
    )

    row = (i // cols) + 1
    col = (i % cols) + 1
    fig.add_trace(heatmap, row=row, col=col)

    # Update x and y axis titles for each subplot
    fig.update_xaxes(title_text="% Outliers", row=row, col=col)
    fig.update_yaxes(title_text="Sampling Fraction", row=row, col=col)

# Update layout
fig.update_layout(
    height=rows * 400,  # Adjust height as needed
    width=cols * 400,  # Adjust width as needed
    title_text="Mean ARI with Varying Outliers and Sampling for Different Noise Levels",
    showlegend=False,
)

# Save the figure as a high-quality image
# pio.write_image(fig, "heatmaps_grid.png", scale=2)  # Adjust scale for higher resolution

# Show the plot (optional)
fig.show()

given k

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Function to read CSV with error handling
def read_csv_with_error_handling(file_path):
    try:
        return pd.read_csv(file_path)
    except pd.errors.ParserError as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame if there's an error


# Step 1: Read the CSV files into DataFrames with error handling
file1 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/results/phyclip_results_given_t.csv"
)
# file2 = read_csv_with_error_handling(
#     "/home/ganesank/project/phytclust/runtime/runtimes_and_memory_phytclust_greedy_single_k.csv"
# )
file3 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/runtimes_and_memory_phytclust_single_k.csv"
)
# Step 1: Read the CSV files into DataFrames with error handling
file4 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/treecluster_time_given_k.csv"
)
# file5 = read_csv_with_error_handling(
#     "/home/ganesank/project/phytclust/runtime/results/phylopart_results.csv"
# )

# Step 3: Add a new column to each DataFrame to indicate the source file
file1["source"] = "phyclip"
# file2["source"] = "phytclust_greedy"
file3["source"] = "phytclust"
file4["source"] = "treecluster"
# file5["source"] = "phylopart"
columns = ["file", "size", "num", "runtime", "peak_memory", "source"]

# Step 4: Reset the index of each DataFrame to ensure unique index values
file1.reset_index(drop=True, inplace=True)
# file2.reset_index(drop=True, inplace=True)
file3.reset_index(drop=True, inplace=True)
file4.reset_index(drop=True, inplace=True)
file5.reset_index(drop=True, inplace=True)

# Step 5: Concatenate the DataFrames
for df in [file1, file3, file4]:
    df["size"] = pd.to_numeric(df["size"], errors="coerce")
    df["num"] = pd.to_numeric(df["num"], errors="coerce")
    df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")
    df["peak_memory"] = pd.to_numeric(df["peak_memory"], errors="coerce")
df = pd.concat([file1, file3, file4], axis=0, ignore_index=True)

df = df.dropna(subset=["size"])

# Step 6: Filter the DataFrame for the specified `size` values
filtered_df = df[df["size"].isin([100, 500, 1000, 2500, 5000, 7500, 10000])]

# Step 7: Group by `size` and calculate the mean `runtime`
grouped_df = filtered_df.groupby(["size", "source"])["runtime"].mean().reset_index()

# Step 8: Plot the results
plt.style.use("seaborn-darkgrid")  # Set a style
plt.figure(figsize=(12, 8))

# Use seaborn color palette
palette = sns.color_palette("husl", len(grouped_df["source"].unique()))

markers = ["o", "s", "D", "^", "v"]  # Different markers for each source

for i, source in enumerate(grouped_df["source"].unique()):
    source_df = grouped_df[grouped_df["source"] == source]
    plt.plot(
        source_df["size"],
        source_df["runtime"],
        marker=markers[i % len(markers)],
        color=palette[i % len(palette)],
        label=source,
        linestyle="-",
        linewidth=2,
        markersize=8,
    )

plt.xlabel("Number of Leaves", fontsize=14)
plt.ylabel("Mean Runtime (seconds)", fontsize=14)
plt.title("Mean Runtime vs Number of Leaves", fontsize=16, fontweight="bold")
plt.yscale("log")  # Set y-axis to logarithmic scale
plt.legend(title="Source", fontsize=12, title_fontsize="13")  # Customize legend
plt.grid(True, which="both", linestyle="--", linewidth=0.5)  # Add grid lines
plt.xticks(
    [100, 500, 1000, 2500, 5000, 7500, 10000], fontsize=12
)  # Ensure x-ticks match the specified size values
plt.yticks(fontsize=12)

# Optionally, add annotations
for i, row in grouped_df.iterrows():
    plt.annotate(
        f"{row['runtime']:.2f}",
        (row["size"], row["runtime"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=10,
    )

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math


# Function to read CSV with error handling
def read_csv_with_error_handling(file_path):
    try:
        return pd.read_csv(file_path)
    except pd.errors.ParserError as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame if there's an error


# Step 1: Read the CSV files into DataFrames with error handling
file1 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/results/phyclip_results_given_t.csv"
)
# file2 = read_csv_with_error_handling(
#     "/home/ganesank/project/phytclust/runtime/runtimes_and_memory_phytclust_greedy_single_k.csv"
# )
file3 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/runtimes_and_memory_phytclust_single_k.csv"
)
file4 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/treecluster_time_given_k.csv"
)
file5 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/results/phylopart_results.csv"
)

# Step 3: Add a new column to each DataFrame to indicate the source file
file1["source"] = "phyclip"
# file2["source"] = "phytclust_greedy"
file3["source"] = "phytclust"
file4["source"] = "treecluster"
file5["source"] = "phylopart"
columns = ["file", "size", "num", "runtime", "peak_memory", "source"]

# Step 4: Reset the index of each DataFrame to ensure unique index values
file1.reset_index(drop=True, inplace=True)
# file2.reset_index(drop=True, inplace=True)
file3.reset_index(drop=True, inplace=True)
file4.reset_index(drop=True, inplace=True)
file5.reset_index(drop=True, inplace=True)

# Step 5: Concatenate the DataFrames
for df in [file1, file3, file4, file5]:
    df["size"] = pd.to_numeric(df["size"], errors="coerce")
    df["num"] = pd.to_numeric(df["num"], errors="coerce")
    df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")
    df["peak_memory"] = pd.to_numeric(df["peak_memory"], errors="coerce")
df = pd.concat([file1, file3, file4, file5], axis=0, ignore_index=True)

df = df.dropna(subset=["size"])

# Step 6: Filter the DataFrame for the specified `size` values
filtered_df = df[df["size"].isin([100, 500, 1000, 2500, 5000, 7500, 10000])]

# Step 7: Group by `size` and calculate the mean `peak_memory`
grouped_df = filtered_df.groupby(["size", "source"])["peak_memory"].mean().reset_index()

# Step 8: Plot the results
plt.style.use("seaborn-darkgrid")  # Set a style
plt.figure(figsize=(12, 8))

# Use seaborn color palette
palette = sns.color_palette("husl", len(grouped_df["source"].unique()))

markers = ["o", "s", "D", "^", "v"]  # Different markers for each source

for i, source in enumerate(grouped_df["source"].unique()):
    source_df = grouped_df[grouped_df["source"] == source]
    plt.plot(
        source_df["size"],
        source_df["peak_memory"],
        marker=markers[i % len(markers)],
        color=palette[i % len(palette)],
        label=source,
        linestyle="-",
        linewidth=2,
        markersize=8,
    )

plt.xlabel("Number of Leaves", fontsize=14)
plt.ylabel("Mean Memory (MB)", fontsize=14)
plt.title("Mean Memory utilized vs Number of Leaves", fontsize=16, fontweight="bold")
plt.yscale("log")  # Set y-axis to logarithmic scale
plt.legend(title="Source", fontsize=12, title_fontsize="13")  # Customize legend
plt.grid(True, which="both", linestyle="--", linewidth=0.5)  # Add grid lines
plt.xticks(
    [100, 500, 1000, 2500, 5000, 7500, 10000], fontsize=12
)  # Ensure x-ticks match the specified size values
plt.yticks(fontsize=12)

# Optionally, add annotations
for i, row in grouped_df.iterrows():
    plt.annotate(
        f"{math.ceil(row['peak_memory'])}",
        (row["size"], row["peak_memory"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=10,
    )

plt.tight_layout()
plt.show()

all_k

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Function to read CSV with error handling
def read_csv_with_error_handling(file_path):
    try:
        return pd.read_csv(file_path)
    except pd.errors.ParserError as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame if there's an error


# Step 1: Read the CSV files into DataFrames with error handling
# file1 = read_csv_with_error_handling(
#     "/home/ganesank/project/phytclust/runtime/results/phyclip_results_pf.csv"
# )
# file2 = read_csv_with_error_handling(
#     "/home/ganesank/project/phytclust/runtime/runtimes_and_memory_phytclust_greedy_single_k.csv"
# )
file3 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/runtimes_and_memory_phytclust_all_k.csv"
)
# Step 1: Read the CSV files into DataFrames with error handling
file4 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/treecluster_time_range_k.csv"
)
file5 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/autophy_runtime_results.csv"
)

# Step 3: Add a new column to each DataFrame to indicate the source file
# file1["source"] = "phyclip"
# file2["source"] = "phytclust_greedy"
file3["source"] = "phytclust"
file4["source"] = "treecluster"
file5["source"] = "autophy"
columns = ["file", "size", "num", "runtime", "peak_memory", "source"]
# Step 4: Reset the index of each DataFrame to ensure unique index values
# file1.reset_index(drop=True, inplace=True)
# file2.reset_index(drop=True, inplace=True)
file3.reset_index(drop=True, inplace=True)
file4.reset_index(drop=True, inplace=True)
file5.reset_index(drop=True, inplace=True)

# Step 5: Concatenate the DataFrames
for df in [file3, file4, file5]:
    df["size"] = pd.to_numeric(df["size"], errors="coerce")
    df["num"] = pd.to_numeric(df["num"], errors="coerce")
    df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")
    df["peak_memory"] = pd.to_numeric(df["peak_memory"], errors="coerce")
df = pd.concat([file3, file4, file5], axis=0, ignore_index=True)

df = df.dropna(subset=["size"])

# Step 6: Filter the DataFrame for the specified `size` values
filtered_df = df[df["size"].isin([100, 500, 1000, 2500, 5000, 7500, 10000])]

# Step 7: Group by `size` and calculate the mean `runtime`
grouped_df = filtered_df.groupby(["size", "source"])["runtime"].mean().reset_index()

# Step 8: Plot the results
plt.style.use("seaborn-darkgrid")  # Set a style
plt.figure(figsize=(12, 8))

# Use seaborn color palette
palette = sns.color_palette("husl", len(grouped_df["source"].unique()))

markers = ["o", "s", "D", "^", "v"]  # Different markers for each source

for i, source in enumerate(grouped_df["source"].unique()):
    source_df = grouped_df[grouped_df["source"] == source]
    plt.plot(
        source_df["size"],
        source_df["runtime"],
        marker=markers[i % len(markers)],
        color=palette[i % len(palette)],
        label=source,
        linestyle="-",
        linewidth=2,
        markersize=8,
    )

plt.xlabel("Number of Leaves", fontsize=14)
plt.ylabel("Mean Runtime (seconds)", fontsize=14)
plt.title("Mean Runtime vs Number of Leaves", fontsize=16, fontweight="bold")
plt.yscale("log")  # Set y-axis to logarithmic scale
plt.legend(title="Source", fontsize=12, title_fontsize="13")  # Customize legend
plt.grid(True, which="both", linestyle="--", linewidth=0.5)  # Add grid lines
plt.xticks(
    [100, 500, 1000, 2500, 5000, 7500, 10000], fontsize=12
)  # Ensure x-ticks match the specified size values
plt.yticks(fontsize=12)

# Optionally, add annotations
for i, row in grouped_df.iterrows():
    plt.annotate(
        f"{row['runtime']:.2f}",
        (row["size"], row["runtime"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=10,
    )

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# Function to read CSV with error handling
def read_csv_with_error_handling(file_path):
    try:
        return pd.read_csv(file_path)
    except pd.errors.ParserError as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()  # Return an empty DataFrame if there's an error


# Step 1: Read the CSV files into DataFrames with error handling
# file1 = read_csv_with_error_handling(
#     "/home/ganesank/project/phytclust/runtime/results/phyclip_results_pf.csv"
# )
# file2 = read_csv_with_error_handling(
#     "/home/ganesank/project/phytclust/runtime/runtimes_and_memory_phytclust_greedy_single_k.csv"
# )
file3 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/runtimes_and_memory_phytclust_all_k.csv"
)
# Step 1: Read the CSV files into DataFrames with error handling
file4 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/treecluster_time_range_k.csv"
)
file5 = read_csv_with_error_handling(
    "/home/ganesank/project/phytclust/runtime/autophy_runtime_results.csv"
)

# Step 3: Add a new column to each DataFrame to indicate the source file
# file1["source"] = "phyclip"
# file2["source"] = "phytclust_greedy"
file3["source"] = "phytclust"
file4["source"] = "treecluster"
file5["source"] = "autophy"
columns = ["file", "size", "num", "runtime", "peak_memory", "source"]
# Step 4: Reset the index of each DataFrame to ensure unique index values
# file1.reset_index(drop=True, inplace=True)
# file2.reset_index(drop=True, inplace=True)
file3.reset_index(drop=True, inplace=True)
file4.reset_index(drop=True, inplace=True)
file5.reset_index(drop=True, inplace=True)

# Step 5: Concatenate the DataFrames
for df in [file3, file4, file5]:
    df["size"] = pd.to_numeric(df["size"], errors="coerce")
    df["num"] = pd.to_numeric(df["num"], errors="coerce")
    df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")
    df["peak_memory"] = pd.to_numeric(df["peak_memory"], errors="coerce")
df = pd.concat([file3, file4, file5], axis=0, ignore_index=True)

df = df.dropna(subset=["size"])

# Step 6: Filter the DataFrame for the specified `size` values
filtered_df = df[df["size"].isin([100, 500, 1000, 2500, 5000, 7500, 10000])]

# Step 7: Group by `size` and calculate the mean `runtime`
grouped_df = filtered_df.groupby(["size", "source"])["peak_memory"].mean().reset_index()

# Step 8: Plot the results
plt.style.use("seaborn-darkgrid")  # Set a style
plt.figure(figsize=(12, 8))

# Use seaborn color palette
palette = sns.color_palette("husl", len(grouped_df["source"].unique()))

markers = ["o", "s", "D", "^", "v"]  # Different markers for each source

for i, source in enumerate(grouped_df["source"].unique()):
    source_df = grouped_df[grouped_df["source"] == source]
    plt.plot(
        source_df["size"],
        source_df["peak_memory"],
        marker=markers[i % len(markers)],
        color=palette[i % len(palette)],
        label=source,
        linestyle="-",
        linewidth=2,
        markersize=8,
    )

plt.xlabel("Number of Leaves", fontsize=14)
plt.ylabel("Mean Peak Memory (MB)", fontsize=14)
plt.title("Mean Peak Memory vs Number of Leaves", fontsize=16, fontweight="bold")
plt.yscale("log")  # Set y-axis to logarithmic scale
plt.legend(title="Source", fontsize=12, title_fontsize="13")  # Customize legend
plt.grid(True, which="both", linestyle="--", linewidth=0.5)  # Add grid lines
plt.xticks(
    [100, 500, 1000, 2500, 5000, 7500, 10000], fontsize=12
)  # Ensure x-ticks match the specified size values
plt.yticks(fontsize=12)

# Optionally, add annotations
for i, row in grouped_df.iterrows():
    plt.annotate(
        f"{row['peak_memory']:.2f}",
        (row["size"], row["peak_memory"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=10,
    )

plt.tight_layout()
plt.show()